# UniDAC metric depth on the project cameras

This Colab notebook runs UniDAC on the calibrated G1_A fisheye or ZED_B perspective images. It saves metric depth in metres, a valid-region mask, a visualization, and reproducibility/timing metadata to Google Drive.

Before running: choose **Runtime → Change runtime type → GPU**. Then run the cells from top to bottom. The one-frame smoke test is the default; batch processing is opt-in at the end.

The official UniDAC source is pinned to commit 9ddfc1f for reproducibility. The released model checkpoint is about 1.4 GB, so the first setup takes longer than later runs.

## 1. Install the lightweight UniDAC runtime dependencies

Colab supplies PyTorch and CUDA. This cell installs the remaining packages required by the official UniDAC inference code. If Colab asks for a runtime restart after installation, restart it and run all cells again.

In [ ]:
import subprocess
import sys

packages = [
    "numpy<2",
    "opencv-python-headless==4.11.0.86",
    "einops>=0.6",
    "timm>=0.9",
    "huggingface_hub>=0.20",
    "matplotlib>=3.8",
    "scipy>=1.10",
]
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *packages])
print("UniDAC runtime dependencies are ready.")

## 2. Experiment settings

Usually only DATA_DIR and OUTPUT_DIR need changing. DATA_DIR must contain intrinsic.json, extrinsics.json, and the recording folders.

In [ ]:
from pathlib import Path

PROJECT_REPO_URL = "https://github.com/esthy13/monocular-depth-estimation.git"
PROJECT_REF = "depth-pipeline"
UNIDAC_COMMIT = "9ddfc1f4cea68e08273ec9bca037f2ef9e1aa90e"

PROJECT_DIR = Path("/content/monocular-depth-estimation")
UNIDAC_DIR = PROJECT_DIR / "third_party" / "UniDAC"

# Change these paths to match your Google Drive folders.
DATA_DIR = Path("/content/drive/MyDrive/cv_project_data")
OUTPUT_DIR = Path("/content/drive/MyDrive/cv_project_outputs/unidac")
CHECKPOINT_CACHE_DIR = Path("/content/drive/MyDrive/cv_project_cache/unidac")

# One-frame smoke test.
RECORDING = "recording1"
SENSOR = "G1_A"  # G1_A = fisheye; ZED_B = perspective
IMAGE_INDEX = 0

# For a stable speed benchmark later, use WARMUP_RUNS=2 and TIMED_RUNS=10.
WARMUP_RUNS = 0
TIMED_RUNS = 1

## 3. Mount Drive and verify the GPU/data

The notebook intentionally stops if CUDA is unavailable, because UniDAC's DINOv3-L backbone is not practical for this workflow on Colab CPU.

In [ ]:
from google.colab import drive

drive.mount("/content/drive")

import torch

if not torch.cuda.is_available():
    raise RuntimeError(
        "CUDA is not available. In Colab choose Runtime → Change runtime type → GPU, "
        "then reconnect and run all cells again."
    )

if not (DATA_DIR / "intrinsic.json").is_file():
    raise FileNotFoundError(
        f"Could not find {DATA_DIR / 'intrinsic.json'}. Upload cv_project_data to Drive "
        "or edit DATA_DIR in the settings cell."
    )

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
CHECKPOINT_CACHE_DIR.mkdir(parents=True, exist_ok=True)
print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"PyTorch: {torch.__version__}; CUDA runtime: {torch.version.cuda}")
print(f"Data: {DATA_DIR}")
print(f"Outputs: {OUTPUT_DIR}")

## 4. Fetch the project and pinned UniDAC source

Existing project files are only fast-forwarded; local changes in the temporary Colab checkout are never overwritten. UniDAC is downloaded from the pinned GitHub source archive, avoiding large demo assets.

In [ ]:
import shutil
import subprocess
import tarfile
import urllib.request

def run_command(arguments):
    subprocess.check_call([str(item) for item in arguments])

if not (PROJECT_DIR / ".git").is_dir():
    run_command([
        "git", "clone", "--branch", PROJECT_REF, "--single-branch",
        PROJECT_REPO_URL, PROJECT_DIR,
    ])
else:
    run_command(["git", "-C", PROJECT_DIR, "fetch", "origin", PROJECT_REF])
    run_command(["git", "-C", PROJECT_DIR, "checkout", PROJECT_REF])
    run_command(["git", "-C", PROJECT_DIR, "merge", "--ff-only", f"origin/{PROJECT_REF}"])

project_commit = subprocess.check_output(
    ["git", "-C", PROJECT_DIR, "rev-parse", "HEAD"], text=True
).strip()

unidac_marker = UNIDAC_DIR / ".pinned_commit"
installed_unidac_commit = (
    unidac_marker.read_text().strip() if unidac_marker.is_file() else None
)
if UNIDAC_DIR.exists() and installed_unidac_commit != UNIDAC_COMMIT:
    raise RuntimeError(
        f"{UNIDAC_DIR} already exists but is not the requested pinned source. "
        "Start a fresh Colab runtime or remove only that directory and rerun this cell."
    )

if not UNIDAC_DIR.exists():
    archive_path = Path("/content/unidac-source.tar.gz")
    staging_dir = Path("/content/unidac-source")
    if staging_dir.exists():
        shutil.rmtree(staging_dir)
    staging_dir.mkdir(parents=True)
    archive_url = f"https://github.com/girish1511/UniDAC/archive/{UNIDAC_COMMIT}.tar.gz"
    print("Downloading pinned UniDAC source ...")
    urllib.request.urlretrieve(archive_url, archive_path)
    with tarfile.open(archive_path, "r:gz") as archive:
        archive.extractall(staging_dir)
    extracted_dir = next(path for path in staging_dir.iterdir() if path.is_dir())
    UNIDAC_DIR.parent.mkdir(parents=True, exist_ok=True)
    shutil.move(str(extracted_dir), str(UNIDAC_DIR))
    unidac_marker.write_text(UNIDAC_COMMIT + "\n")
    archive_path.unlink(missing_ok=True)
    shutil.rmtree(staging_dir)

print(f"Project commit: {project_commit}")
print(f"UniDAC commit:  {UNIDAC_COMMIT}")

## 5. Cache the official checkpoint in Drive

The checkpoint is kept in Drive so later Colab sessions do not download it again. A temporary local copy is used for faster model loading.

In [ ]:
from huggingface_hub import hf_hub_download

drive_checkpoint = Path(hf_hub_download(
    repo_id="girish1511/UniDAC",
    filename="unidac.pt",
    local_dir=str(CHECKPOINT_CACHE_DIR),
))
local_checkpoint = Path("/content/checkpoints/unidac.pt")
local_checkpoint.parent.mkdir(parents=True, exist_ok=True)
if (
    not local_checkpoint.is_file()
    or local_checkpoint.stat().st_size != drive_checkpoint.stat().st_size
):
    print("Copying the checkpoint from Drive to the Colab VM ...")
    shutil.copy2(drive_checkpoint, local_checkpoint)

print(f"Checkpoint: {local_checkpoint}")
print(f"Size: {local_checkpoint.stat().st_size / 1024**3:.2f} GiB")

## 6. Load UniDAC once

This is the memory-heavy step. The adapter removes the inaccessible DINOv3 path embedded in the released config, then loads every parameter from the complete official UniDAC checkpoint.

In [ ]:
import sys

for import_path in (PROJECT_DIR, UNIDAC_DIR):
    if str(import_path) not in sys.path:
        sys.path.insert(0, str(import_path))

from src.depth_models import UniDACDepth

model = UniDACDepth(
    device="cuda",
    repo_dir=str(UNIDAC_DIR),
    checkpoint_path=str(local_checkpoint),
)
model.load()

## 7. Helpers for calibrated inference and reproducible outputs

In [ ]:
import csv
import json
import math
import time

import cv2
import numpy as np

from src.utils import (
    create_fisheye_valid_mask,
    find_rgb_images,
    intrinsics_to_dac_cam_params,
    load_intrinsics,
    parse_timestamp,
    save_depth_visualization,
    save_mask_visualization,
)

intrinsics = load_intrinsics(DATA_DIR / "intrinsic.json")

def camera_geometry(sensor, image_width):
    camera = intrinsics[sensor]
    camera_parameters = intrinsics_to_dac_cam_params(sensor, intrinsics)
    if camera.get("model") == "fisheye":
        crop_wfov = 180.0
    else:
        focal_x = float(camera["K"][0][0])
        crop_wfov = math.degrees(2.0 * math.atan(image_width / (2.0 * focal_x)))
    return camera_parameters, crop_wfov

def timed_prediction(image, warmup_runs=0, timed_runs=1):
    if timed_runs < 1:
        raise ValueError("timed_runs must be at least 1")
    for _ in range(warmup_runs):
        model.predict(image)
    torch.cuda.synchronize()
    timings_ms = []
    depth = None
    for _ in range(timed_runs):
        start = time.perf_counter()
        depth = model.predict(image)
        torch.cuda.synchronize()
        timings_ms.append((time.perf_counter() - start) * 1000.0)
    return depth, timings_ms

def process_frame(recording, sensor, image_index, warmup_runs=0, timed_runs=1):
    images = find_rgb_images(DATA_DIR, sensor_name=sensor, recording=recording)
    if not 0 <= image_index < len(images):
        raise IndexError(
            f"image_index={image_index} is outside 0..{len(images) - 1} for "
            f"{recording}/{sensor}"
        )
    image_path = images[image_index]
    image = cv2.imread(str(image_path))
    if image is None:
        raise RuntimeError(f"OpenCV could not read {image_path}")

    camera_parameters, crop_wfov = camera_geometry(sensor, image.shape[1])
    model.set_camera(camera_parameters, crop_wfov)
    depth, timings_ms = timed_prediction(image, warmup_runs, timed_runs)

    valid_mask = np.isfinite(depth) & (depth > 0)
    if intrinsics[sensor].get("model") == "fisheye":
        K = intrinsics[sensor]["K"]
        lens_mask = create_fisheye_valid_mask(image, center=(K[0][2], K[1][2]))
        valid_mask &= lens_mask.astype(bool)
    if not np.any(valid_mask):
        raise RuntimeError("UniDAC returned no valid positive depth pixels.")
    depth = depth.astype(np.float32, copy=True)
    depth[~valid_mask] = np.nan

    frame_dir = OUTPUT_DIR / recording / sensor
    frame_dir.mkdir(parents=True, exist_ok=True)
    stem = f"{recording}_{sensor}_{image_index:06d}"
    rgb_path = frame_dir / f"{stem}_rgb.jpg"
    raw_path = frame_dir / f"{stem}_depth_raw.npy"
    mask_path = frame_dir / f"{stem}_mask.png"
    visualization_path = frame_dir / f"{stem}_depth.png"
    metadata_path = frame_dir / f"{stem}_metadata.json"

    cv2.imwrite(str(rgb_path), image)
    np.save(raw_path, depth)
    save_mask_visualization(valid_mask.astype(np.uint8), mask_path)
    save_depth_visualization(
        depth,
        visualization_path,
        rgb_image=image,
        valid_mask=valid_mask,
        invert=True,
    )

    valid_depth = depth[valid_mask]
    try:
        timestamp_seconds = parse_timestamp(image_path)
    except ValueError:
        timestamp_seconds = None
    metadata = {
        "model": "UniDAC",
        "checkpoint_repo": "girish1511/UniDAC",
        "checkpoint_file": "unidac.pt",
        "unidac_commit": UNIDAC_COMMIT,
        "project_commit": project_commit,
        "depth_definition": "metric Euclidean ray distance",
        "depth_unit": "metre",
        "recording": recording,
        "sensor": sensor,
        "image_index": image_index,
        "image_file": image_path.name,
        "timestamp_seconds": timestamp_seconds,
        "image_width": image.shape[1],
        "image_height": image.shape[0],
        "camera_model": intrinsics[sensor].get("model"),
        "crop_wfov_degrees": crop_wfov,
        "gpu": torch.cuda.get_device_name(0),
        "torch_version": torch.__version__,
        "cuda_version": torch.version.cuda,
        "warmup_runs": warmup_runs,
        "timed_runs": timed_runs,
        "timing_scope": "preprocess + model + camera back-projection",
        "timings_ms": timings_ms,
        "median_time_ms": float(np.median(timings_ms)),
        "valid_fraction": float(valid_mask.mean()),
        "valid_depth_min_m": float(valid_depth.min()),
        "valid_depth_median_m": float(np.median(valid_depth)),
        "valid_depth_max_m": float(valid_depth.max()),
    }
    metadata_path.write_text(json.dumps(metadata, indent=2) + "\n")
    return metadata, visualization_path

print(f"Loaded calibration for: {sorted(intrinsics)}")

## 8. Run the one-frame smoke test

This writes five files to Drive: RGB, raw float32 depth, valid mask, visualization, and JSON metadata.

In [ ]:
from IPython.display import Image as DisplayImage
from IPython.display import display

metadata, visualization_path = process_frame(
    RECORDING,
    SENSOR,
    IMAGE_INDEX,
    warmup_runs=WARMUP_RUNS,
    timed_runs=TIMED_RUNS,
)
print(json.dumps(metadata, indent=2))
display(DisplayImage(filename=str(visualization_path)))

## 9. Optional resumable batch

Set RUN_BATCH to True only after the smoke test looks correct. Each finished frame has its own metadata file; rerunning the cell skips completed frames unless OVERWRITE is True. FRAME_STEP can sample every nth frame for an initial evaluation.

In [ ]:
RUN_BATCH = False
BATCH_RECORDINGS = ["recording1"]
BATCH_SENSORS = ["G1_A", "ZED_B"]
FRAME_STEP = 10
MAX_FRAMES_PER_SENSOR = 50  # None means all selected frames
OVERWRITE = False
BATCH_WARMUP_RUNS = 0
BATCH_TIMED_RUNS = 1

if not RUN_BATCH:
    print("Batch is disabled. Set RUN_BATCH=True when ready.")
else:
    completed = 0
    skipped = 0
    for recording in BATCH_RECORDINGS:
        for sensor in BATCH_SENSORS:
            images = find_rgb_images(DATA_DIR, sensor_name=sensor, recording=recording)
            selected_indices = list(range(0, len(images), FRAME_STEP))
            if MAX_FRAMES_PER_SENSOR is not None:
                selected_indices = selected_indices[:MAX_FRAMES_PER_SENSOR]
            for image_index in selected_indices:
                stem = f"{recording}_{sensor}_{image_index:06d}"
                frame_dir = OUTPUT_DIR / recording / sensor
                expected = [
                    frame_dir / f"{stem}_depth_raw.npy",
                    frame_dir / f"{stem}_depth.png",
                    frame_dir / f"{stem}_metadata.json",
                ]
                if not OVERWRITE and all(path.is_file() for path in expected):
                    skipped += 1
                    continue
                metadata, _ = process_frame(
                    recording,
                    sensor,
                    image_index,
                    warmup_runs=BATCH_WARMUP_RUNS,
                    timed_runs=BATCH_TIMED_RUNS,
                )
                completed += 1
                print(
                    f"[{completed}] {recording}/{sensor}/{image_index}: "
                    f"{metadata['median_time_ms']:.1f} ms"
                )

    summary_fields = [
        "recording", "sensor", "image_index", "image_file",
        "timestamp_seconds", "gpu", "median_time_ms", "valid_fraction",
        "valid_depth_min_m", "valid_depth_median_m", "valid_depth_max_m",
        "project_commit", "unidac_commit",
    ]
    summary_rows = []
    for metadata_path in sorted(OUTPUT_DIR.rglob("*_metadata.json")):
        record = json.loads(metadata_path.read_text())
        summary_rows.append({field: record.get(field) for field in summary_fields})
    summary_path = OUTPUT_DIR / "unidac_summary.csv"
    with summary_path.open("w", newline="") as summary_file:
        writer = csv.DictWriter(summary_file, fieldnames=summary_fields)
        writer.writeheader()
        writer.writerows(summary_rows)
    print(
        f"Batch complete: {completed} processed, {skipped} skipped. "
        f"Summary: {summary_path}"
    )

## Benchmark note

For the semester report, use the same GPU type, camera, input frames, warm-up count, and timed-run count for every model. The saved timing is end-to-end (projection, neural inference, and back-projection), which is the relevant latency for the full 3D pipeline. Keep the JSON metadata alongside every raw depth array.